In [ ]:
from pyrokinetics import Pyro, PyroScan, template_dir
import os
import pathlib
from typing import Union
import numpy as np
import copy
import subprocess
from pathlib import Path
from pyrokinetics.diagnostics.gs2_gp import gs2_gp
import run_simulations
import json

REPO_ROOT = Path.cwd().resolve().parent

GYRO_DATA = Path("/home/Felix/Documents/Physics_Work/Project_Codes/Gyrokinetic_Simulations/")

PROJECT_NAME = "Beta_Prime_Scan"

In [ ]:
def load_gs2_pyroscan(step_case, project, name="gs2"):
    json_path = (
        GYRO_DATA
        / "GS2"
        / "Runs"
        / project
        / step_case
        / f"parameter_scan_{name}"
        / "pyroscan.json"
    )
    in_loc = GYRO_DATA / "GS2" / "Templates" / step_case / "gs2.in"
    pyro_object = Pyro(gk_file=in_loc, gk_code="GS2")
    return PyroScan(pyro=pyro_object, pyroscan_json=json_path)
    # Add function to enforce consistent beta prime
    pyro_scan_tglf_ML.add_parameter_key(
        parameter_key="beta", parameter_attr="numerics", parameter_location=["beta"]
    )

    # Add function to tglf
    pyro_scan_tglf_ML.add_parameter_func(param_2, enforce_beta_prime, param_2_kwargs)

    # Create scan directory and write input files
    pyro_scan_tglf_ML.write(
        file_name="input.tglf",
        base_directory=GYRO_DATA / "TGLF" / base_out_loc / "parameter_scan_tglf_ML",
        template_file=None,
    )

    return pyro_scan_gs2


def load_gs2_pyroscan(step_case, project, name="gs2"):
    json_path = (
        GYRO_DATA
        / "GS2"
        / "Runs"
        / project
        / step_case
        / f"parameter_scan_{name}"
        / "pyroscan.json"
    )
    in_loc = GYRO_DATA / "GS2" / "Templates" / step_case / "gs2.in"
    pyro_object = Pyro(gk_file=in_loc, gk_code="GS2")
    return PyroScan(pyro=pyro_object, pyroscan_json=json_path)


def load_tglf_pyroscan(step_case, project, name="tglf"):
    json_path = (
        GYRO_DATA
        / "TGLF"
        / "Runs"
        / project
        / step_case
        / f"parameter_scan_{name}"
        / "pyroscan.json"
    )
    in_loc = GYRO_DATA / "GS2" / "Templates" / step_case / "gs2.in"
    pyro_object = Pyro(gk_file=in_loc, gk_code="GS2")
    pyro_object.gk_code = "TGLF"
    return PyroScan(pyro=pyro_object, pyroscan_json=json_path)

In [ ]:
step_case = "R1"

# Generate the input files
# Read_from_gs2(step_case)

gs2_scan_names = ["gs2"]
tglf_scan_names = ["tglf","tglf_F", "tglf_M"]
gs2_scans = []
tglf_scans = []
for name in gs2_scan_names:
    gs2_scans.append(load_gs2_pyroscan(step_case, PROJECT_NAME, name=name))
for name in tglf_scan_names:
    tglf_scans.append(load_tglf_pyroscan(step_case, PROJECT_NAME, name=name))


In [ ]:
# load models
models_path = "/home/Felix/Documents/Physics_Work/Project_Codes/8d_Up2/"


models = [
    "growth_rate_log",
    "mode_frequency_log",
]

In [ ]:
gs2_scans[0].load_gk_output()
gs2_data = gs2_scans[0].gk_output

tglf_data = []
for tglf_scan  in tglf_scans:
    tglf_scan.load_gk_output()
    tglf_data.append(tglf_scan.gk_output)



In [ ]:
Gaussian_model = gs2_gp(pyro=gs2_scans[0], models_path=models_path, models=models).gk_output
print(Gaussian_model)

In [ ]:
from typing import Dict, List, Any, Optional
from pyrokinetics import Pyro, PyroScan
import numpy as np
import torch
from pathlib import Path
from pyrokinetics.diagnostics.gs2_gp import gs2_gp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
def Ground_Truth_2d(
    ground_truth_run,
    runs,
    names,
    plot_location,
    Gaussian=False,
    Gaussian_Model=None,
    parameter_1_range=None,
    parameter_2_range=None,
):
    growth_rate_list = []
    mode_freq_list = []
    data = ground_truth_run
    ground_truth_growth_rate = data["growth_rate"]
    ground_truth_mode_freq = data["mode_frequency"]
    print(runs)
    for run in runs:
        print(run)
        print(names)
        data = run
        growth_rate_list.append(data["growth_rate"])
        mode_freq_list.append(data["mode_frequency"])
    # Compute absolute model error relative to ground truth
    difference_growth_list = [
        growth_rate - ground_truth_growth_rate for growth_rate in growth_rate_list
    ]

    difference_mode_list = [
        mode_frequency - ground_truth_mode_freq for mode_frequency in mode_freq_list
    ]
    if Gaussian:
        Gaussian_growth_rate = Gaussian_Model["growth_rate_log_M52"]
        Gaussian_mode_frequency = Gaussian_Model["mode_frequency_log_M32"]

        # GP central predicted values (not bounds)
        GP_growth_value = Gaussian_growth_rate.sel(output="value")
        GP_mode_value   = Gaussian_mode_frequency.sel(output="value")

        # Compute GP - Ground Truth differences
        gp_diff_growth = GP_growth_value - ground_truth_growth_rate
        gp_diff_mode   = GP_mode_value   - ground_truth_mode_freq

        GP_growth_value_max = Gaussian_growth_rate.sel(output="max_value")
        GP_mode_value_max   = Gaussian_mode_frequency.sel(output="max_value")

        # Compute GP - Ground Truth differences
        gp_diff_growth_max = GP_growth_value - ground_truth_growth_rate
        gp_diff_mode_max   = GP_mode_value   - ground_truth_mode_freq

        GP_growth_value_min = Gaussian_growth_rate.sel(output="min_value")
        GP_mode_value_min   = Gaussian_mode_frequency.sel(output="min_value")

        # Compute GP - Ground Truth differences
        gp_diff_growth_min = GP_growth_value - ground_truth_growth_rate
        gp_diff_mode_min   = GP_mode_value   - ground_truth_mode_freq



    for i in range(0, 2):
        major_cord = list(growth_rate_list[0].coords)[i]
        minor_cord = list(growth_rate_list[0].coords)[1 - i]

        list_to_loop_over = growth_rate_list[0].coords[major_cord]
        if i == 0:
            if parameter_1_range != None:
                list_to_loop_over = growth_rate_list[0].coords[major_cord][
                    slice(*parameter_1_range)
                ]
        elif i == 1:
            if parameter_2_range != None:
                list_to_loop_over = growth_rate_list[0].coords[major_cord][
                    slice(*parameter_2_range)
                ]

        n_rows = len(list_to_loop_over)
        n_cols = 3
        aspect_ratio = 2.0  # width:height ratio per subplot
        width = aspect_ratio * n_cols * 3
        height = n_rows * 2.5
        fig = plt.figure(figsize=(width, height))
        gs = gridspec.GridSpec(len(list_to_loop_over), 3, hspace=0, wspace=0.2)
        axes = np.empty((len(list_to_loop_over), 3), dtype=object)

        Plot_Title = f"Plot of Growth Rate and Frequency against {minor_cord} for fixed {major_cord}"

        for j, coor in enumerate(list_to_loop_over):
            # Create subplots
            ax1 = fig.add_subplot(gs[j, 0])
            ax2 = fig.add_subplot(gs[j, 1])
            axes[j, 0] = ax1
            axes[j, 1] = ax2

            # Plot data
            for growth_rate, mode_frequency, name in zip(
                growth_rate_list, mode_freq_list, names
            ):
                ax1.plot(
                    growth_rate[minor_cord].values,  # <-- add .values
                    growth_rate.sel({major_cord: coor}).sel(mode=0),
                    label=rf"{name}",
                    linestyle="--",
                )
                ax2.plot(
                    mode_frequency[minor_cord].values,  # <-- add .values
                    mode_frequency.sel({major_cord: coor}).sel(mode=0),
                    label=rf"{name}",
                    linestyle="--",

                )

            # plot Ground Truth Values
            ax1.plot(
                ground_truth_growth_rate[minor_cord].values,  # <-- add .values
                ground_truth_growth_rate.sel({major_cord: coor}),
                label=rf"GS2",
                color = "black"
            )
            ax2.plot(
                ground_truth_mode_freq[minor_cord].values,  # <-- add .values
                ground_truth_mode_freq.sel({major_cord: coor}),
                label=rf"GS2",
                color = "black"            
            )

            ax3 = fig.add_subplot(gs[j, 2])
            axes[j, 2] = ax3
            
            for diff_growth, diff_mode, name in zip(
                difference_growth_list, difference_mode_list, names
            ):
                ax3.plot(
                    diff_growth[minor_cord],
                    diff_growth.sel({major_cord: coor}).sel(mode=0),
                    label=rf"{name}",
                    linestyle="--",
                    linewidth=2,
                )
            if Gaussian:
                # Use consistent x-axis for plot and fill
                x_data = gp_diff_growth[minor_cord].values
                y_data = gp_diff_growth.sel({major_cord: coor}).values
                y_min = gp_diff_growth_min.sel({major_cord: coor}).values
                y_max = gp_diff_growth_max.sel({major_cord: coor}).values
                
                ax3.fill_between(
                    x_data,  # <-- same x as plot line
                    y_min,
                    y_max,
                    color="red",
                    alpha=0.2,
                    label="_nolegend_",  # hide from legend (optional)
                )
                
                ax3.plot(
                    x_data,
                    y_data,
                    label="GS2_GP",
                    color="red",
                    linestyle=":",
                    linewidth=2,
                

            # Reference: Zero = perfect agreement
            ax3.axhline(0, color="black", linewidth=1)

            # Set title only for the first row (j == 0)
            if j == 0:
                ax1.set_title("Growth Rates", fontsize=16)
                ax2.set_title("Mode Frequencies", fontsize=16)
                ax3.set_title("Growth Rate Error", fontsize=16)

            if Gaussian:
                # Extract data
                x1 = Gaussian_growth_rate[minor_cord].values  # <-- add .values
                y1 = Gaussian_growth_rate.sel({major_cord: coor}).sel(output="value")
                y1_max = Gaussian_growth_rate.sel({major_cord: coor}).sel(
                    output="max_value"
                )
                y1_min = Gaussian_growth_rate.sel({major_cord: coor}).sel(
                    output="min_value"
                )

                x2 = Gaussian_mode_frequency[minor_cord].values  # <-- add .values
                y2 = Gaussian_mode_frequency.sel({major_cord: coor}).sel(output="value")
                y2_max = Gaussian_mode_frequency.sel({major_cord: coor}).sel(
                    output="max_value"
                )
                y2_min = Gaussian_mode_frequency.sel({major_cord: coor}).sel(
                    output="min_value"
                )

                # Plot with shaded error region and dotted central line
                # --- Growth rate ---
                ax1.plot(
                    x1, y1, linestyle=":", color="red", label="GS2_GP"
                )  # dotted central line
                ax1.fill_between(
                    x1, y1_min, y1_max, color="red", alpha=0.2
                )  # light shading

                # --- Mode frequency ---
                ax2.plot(
                    x2, y2, linestyle=":", color="red", label="GS2_GP"
                )  # dotted central line
                ax2.fill_between(x2, y2_min, y2_max, color="red", alpha=0.2)

            ax1.grid(True)
            ax2.grid(True)

            # Set axis labels for bottom row only
            if j == len(list_to_loop_over) - 1:  # last row
                ax1.set_xlabel(f"{minor_cord}", fontsize=12)
                ax2.set_xlabel(f"{minor_cord}", fontsize=12)
                ax3.set_xlabel(f"{minor_cord}", fontsize=12)
            

            if j == len(list_to_loop_over) - 2:  # last row
                ax1.set_ylabel("Growth Rate  $a/c_s$", fontsize=12)
                ax2.set_ylabel("Mode Frequency $c_s/a$", fontsize=12)
                ax3.set_ylabel("Error  $a/c_s$", fontsize=12)

            # Row label on right-hand side
                transform=ax2.transAxes,
                va="center",
                ha="left",
                fontsize=14,
            )

        # for i in range(
        #     len(growth_rate_list[0].coords[major_cord]) - 1
        # ):  # all rows except bottom
        #     axes[i, 0].set_xticklabels([])
        #     axes[i, 0].set_xlabel("")
        #     axes[i, 1].set_xticklabels([])
        #     axes[i, 1].set_xlabel("")

        # Only bottom row gets x-axis labels
        # coord_units = getattr(growth_rate_list[0].coords[minor_cord], "units", "dimensionless")
        # axes[-1, 0].set_xlabel(f"{minor_cord} [{coord_units}]")
        # axes[-1, 1].set_xlabel(f"{minor_cord} [{coord_units}]")

        # Layout and title
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.suptitle(step_case, fontsize = 20)

        # Collect all handles and labels from every axis
        handles, labels = [], []
        for ax_row in axes:
            for ax in ax_row:
                h, l = ax.get_legend_handles_labels()
                handles.extend(h)
                labels.extend(l)
                ax.tick_params(labelsize=11)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

        # Deduplicate by label
        unique = dict(zip(labels, handles))

        # Create one legend for the whole figure
        fig.legend(
            unique.values(),
            unique.keys(),
            loc="upper right",  # position above the subplots
            fontsize=16,
        )

        plt.subplots_adjust(top=0.9, bottom=0.1)  # give room for legend and title
        Plot_Name = f"fixed_{major_cord}_from_{list_to_loop_over[0].values}_to_{list_to_loop_over[-1].values}_against_ground_truth"
        if Gaussian:
            Plot_Name += "_with Gaussian"
.5,
                rf"{major_cord}={coor:.2f}",
                transform=ax2.transAxes,
                va="center",
                ha="left",
                fontsize=14,
            )

        # for i in range(
        #     len(growth_rate_list[0].coords[major_cord]) - 1
        # ):  # all rows except bottom
        #     axes[i, 0].set_xticklabels([])
        #     axes[i, 0].set_xlabel("")
        #     axes[i, 1].set_xticklabels([])
        #     axes[i, 1].set_xlabel("")

        # Only bottom row gets x-axis labels
        # coord_units = getattr(growth_rate_list[0].coords[minor_cord], "units", "dimensionless")
        # axes[-1, 0].set_xlabel(f"{minor_cord} [{coord_units}]")
        # axes[-1, 1].set_xlabel(f"{minor_cord} [{coord_units}]")

        # Layout and title
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.suptitle(step_case, fontsize = 20)

        # Collect all handles and labels from every axis
        handles, labels = [], []
        for ax_row in axes:
            for ax in ax_row:
                h, l = ax.get_legend_handles_labels()
                handles.extend(h)
                labels.extend(l)
                ax.tick_params(labelsize=11)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

        # Deduplicate by label
        unique = dict(zip(labels, handles))

        # Create one legend for the whole figure
        fig.legend(
            unique.values(),
            unique.keys(),
            loc="upper right",  # position above the subplots
            fontsize=16,
        )

        plt.subplots_adjust(top=0.9, bottom=0.1)  # give room for legend and title
        Plot_Name = f"fixed_{major_cord}_from_{list_to_loop_over[0].values}_to_{list_to_loop_over[-1].values}_against_ground_truth"
        if Gaussian:
            Plot_Name += "_with Gaussian"

        # Save everything in ONE file
        print(f"Saving Figure at location: {plot_location}")
        filename = plot_location / f"{Plot_Name}.png"
        plot_location.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(filename, dpi=300)
        plt.close(fig)
        # Save everything in ONE file
        print(f"Saving Figure at location: {plot_location}")
        filename = plot_location / f"{Plot_Name}.png"
        plot_location.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(filename, dpi=300)
        plt.close(fig)    ax3.text(
                2.15,
                0


In [ ]:
plot_location = REPO_ROOT / "Plots" / PROJECT_NAME / step_case

Ground_Truth_2d(
        gs2_data,
        tglf_data,
        tglf_scan_names,
        plot_location,
        Gaussian=True,
        Gaussian_Model = Gaussian_model,
        parameter_1_range=(1, 4),
    )